In [8]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

file_path = "report.pdf"

loader = PyPDFLoader(file_path)
pages = []

async for page in loader .alazy_load():
    pages.append(page)

In [9]:
print("페이지 수: ", len(pages))
print("첫 번째 페이지 정보: ", pages[0])

페이지 수:  12
첫 번째 페이지 정보:  page_content='가상 데이터 · 투자정보 아님  |  기준일 2026-10-01
01 / 12
DATA LAB / RAG PRACTICE / VERSION 1.0
주가와 섹터 조건에 맞는
주식 찾기 에이전트
RAG 실습용 가상 주식 데이터 보고서
전 종목과 수치는 가상 데이터입니다.
실제 상장기업, 시장 시세, 공시 또는 투자 추천을 나타내지 않습니다. 종목코드는 실습용으로 임의 부여한 6자리
 문자열입니다.
24개 가상 종목
8개 섹터
12페이지
정량 조건 검색
자연어 근거 검색
정답 검증 질의
실습 목표
“반도체 중 5만원 이하”, “방산 중 모멘텀 양수”, “PBR 1배 이하 금융주”를 검색하고, 조건에 맞는 종목과 근거를 제시
하는 에이전트를 구현합니다.
가상 기준일: 2026-10-01 | 작성일: 2026-10-07
외부 자료를 사용하지 않고 교육용으로 구성한 정적 데이터입니다. 문서에 없는 최신 시세나 미래 수익률은 판단할 수 없습니다.' metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-07T09:10:30+09:00', 'author': 'RAG 실습 데이터', 'keywords': '', 'moddate': '2026-10-07T09:10:30+09:00', 'subject': '교육용 가상 종목 24개와 검색 검증 질의', 'title': '주가와 섹터 조건에 맞는 주식 찾기 에이전트 - RAG 가상 데이터', 'trapped': '/False', 'source': 'report.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
docs = text_splitter.split_documents(pages)

In [11]:
print(f"총 {len(docs)}개의 청크 생성 완료")
print("각 청크의 길이: ", [len(i.page_content) for i in docs])

for i in docs:
    print("[메타데이터]", i.metadata)
    print("[내용]", i.page_content)
    print("=" * 100)

총 33개의 청크 생성 완료
각 청크의 길이:  [472, 467, 427, 496, 499, 429, 495, 499, 203, 497, 499, 195, 494, 496, 205, 496, 495, 201, 477, 485, 193, 490, 497, 198, 500, 494, 230, 478, 497, 193, 478, 496, 193]
[메타데이터] {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-07T09:10:30+09:00', 'author': 'RAG 실습 데이터', 'keywords': '', 'moddate': '2026-10-07T09:10:30+09:00', 'subject': '교육용 가상 종목 24개와 검색 검증 질의', 'title': '주가와 섹터 조건에 맞는 주식 찾기 에이전트 - RAG 가상 데이터', 'trapped': '/False', 'source': 'report.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}
[내용] 가상 데이터 · 투자정보 아님  |  기준일 2026-10-01
01 / 12
DATA LAB / RAG PRACTICE / VERSION 1.0
주가와 섹터 조건에 맞는
주식 찾기 에이전트
RAG 실습용 가상 주식 데이터 보고서
전 종목과 수치는 가상 데이터입니다.
실제 상장기업, 시장 시세, 공시 또는 투자 추천을 나타내지 않습니다. 종목코드는 실습용으로 임의 부여한 6자리
 문자열입니다.
24개 가상 종목
8개 섹터
12페이지
정량 조건 검색
자연어 근거 검색
정답 검증 질의
실습 목표
“반도체 중 5만원 이하”, “방산 중 모멘텀 양수”, “PBR 1배 이하 금융주”를 검색하고, 조건에 맞는 종목과 근거를 제시
하는 에이전트를 구현합니다.
가상 기준일: 2026-10-01 | 작성일: 2026-10-07
외부 

In [12]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

DB_PATH = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=DB_PATH,
    collection_name="report_pdf",
)

In [14]:
vectorstore.similarity_search("PBR 1배 이하 금융주", k=3)

[Document(id='9553890b-2323-4c8b-99ee-bf9c9927af53', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'keywords': '', 'source': 'report.pdf', 'moddate': '2026-10-07T09:10:30+09:00', 'creationdate': '2026-10-07T09:10:30+09:00', 'title': '주가와 섹터 조건에 맞는 주식 찾기 에이전트 - RAG 가상 데이터', 'page': 7, 'trapped': '/False', 'total_pages': 12, 'page_label': '8', 'author': 'RAG 실습 데이터', 'subject': '교육용 가상 종목 24개와 검색 검증 질의', 'creator': '(unspecified)'}, page_content='RAG 실습 보고서  /  주가·섹터 조건 검색\n가상 데이터 · 투자정보 아님  |  기준일 2026-10-01\n08 / 12\n07 / SECTOR\n금융\n금융지주, 보험, 증권으로 구성했습니다. PBR 1.00배 종목은 “1배 이하”에는 포함되고 “1배 미만”에서는 제외됩니다.\n든든금융지주 | 990501\n청크 ID: STK_990501 / 섹터: 금융 / 가상 기준일: 2026-10-01\n현재가\n14,500원\n시가총액\n28,600억원\nPER\n4.8배\nPBR\n0.48배\nROE\n10.0%\n52주 고가\n18,000원\n52주 저가\n11,000원\n3개월 수익률\n+7.2%\n배당수익률\n5.8%\n핵심 키워드: 은행, 배당, 자사주\n투자 포인트 (가상): 은행 중심의 가상 금융지주다. 배당 확대와 자사주 소각을 주주환원 시나리오로 설정했다.\n리스크 (가상): 대출 부실 증가와 순이자마진 하락이 이익과 배당 여력을 낮출 수 있다.\n푸른손해보험 | 990502'),
 Document(id='ed86994a-a9